# Issue #5 — XGBoost + 변이 유형 피처 암종 분류

기존 `XGB를 활용한 암종분류.ipynb`의 **데이터 검증 → 공용 5-fold → XGBoost → OOF Macro F1 → 제출** 흐름을 유지하면서, 단순 변이 존재 여부 대신 Issue #5의 도메인 기반 변이 유형 피처를 적용한 새 Notebook입니다.

- 원본 CSV와 기존 XGBoost Notebook은 수정하지 않습니다.
- `SUBCLASS`는 피처 생성에 사용하지 않습니다.
- 기본 `RUN_MODE="explore"`에서는 전처리와 pandas 확인까지만 실행합니다.
- 공식 학습은 `experiment` label이 붙은 Issue 브랜치에서만 `RUN_MODE="experiment"`로 실행합니다.

## 0. 변경한 전처리

기존 baseline의 4,384개 유전자별 mutation presence 대신 다음 30,697개 희소 피처를 사용합니다.

- 샘플별 변이 유전자 수, 전체 변이 수, 복수 변이 유전자 수
- `missense`, `synonymous`, `nonsense`, `frameshift`, `complex` 유형별 개수
- 유전자별 변이 여부
- 유전자 × 변이 유형 indicator
- train/test 결측 셀 indicator

HGVS 문자열과 MANE 재표기는 모델 입력에 사용하지 않습니다. 신뢰할 원 transcript가 없으므로 상대 위치 피처도 제외합니다.

In [11]:
from __future__ import annotations

import json
import os
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display
from scipy import sparse
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "PROJECT_CONTEXT.md").is_file() and (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("open_cancer 저장소 안에서 Notebook을 실행하세요.")


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from open_cancer.constants import CLASS_LABELS, PROBABILITY_COLUMNS
from open_cancer.experiment import resolve_experiment_context
from open_cancer.mutation_features import build_mutation_features
from open_cancer.validation import validate_competition_data, validate_submission

## 1. 실행 설정

`RUN_MODE="explore"`는 모델 점수를 만들지 않습니다. 공식 실행 전에는 Issue #5가 실제 Experiment Issue인지 확인해야 합니다. 일반 task Issue라면 새 Experiment Issue를 만든 뒤 그 번호의 브랜치에서 실행합니다.

In [4]:
SEED = 42
N_SPLITS = 5
RUN_MODE = "explore"  # "explore" 또는 "experiment"
USE_BALANCED_SAMPLE_WEIGHT = True
N_JOBS = max(1, min(8, os.cpu_count() or 1))

experiment_context = resolve_experiment_context(RUN_MODE, cwd=PROJECT_ROOT)
ISSUE_NUMBER = experiment_context.issue_number
EXPERIMENT_ID = experiment_context.experiment_id
RUN_TRAINING = experiment_context.is_experiment

random.seed(SEED)
np.random.seed(SEED)

{
    "run_mode": RUN_MODE,
    "branch": experiment_context.branch,
    "issue_number": ISSUE_NUMBER,
    "experiment_id": EXPERIMENT_ID,
    "training_enabled": RUN_TRAINING,
    "seed": SEED,
    "xgboost_version": xgb.__version__,
    "n_jobs": N_JOBS,
}

{'run_mode': 'explore',
 'branch': 'issue-5-hgvs-protein-normalization',
 'issue_number': 5,
 'experiment_id': None,
 'training_enabled': False,
 'seed': 42,
 'xgboost_version': '3.2.0',
 'n_jobs': 8}

In [5]:
DATA_DIR = PROJECT_ROOT / "data" / "raw"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"
FOLD_PATH = PROJECT_ROOT / "data" / "splits" / "stratified_5fold_seed42.csv"
FEATURE_DIR = PROJECT_ROOT / "data" / "processed" / "mutation_type_features"

{
    "project_root": str(PROJECT_ROOT),
    "train": str(TRAIN_PATH.relative_to(PROJECT_ROOT)),
    "test": str(TEST_PATH.relative_to(PROJECT_ROOT)),
    "fold": str(FOLD_PATH.relative_to(PROJECT_ROOT)),
    "features": str(FEATURE_DIR.relative_to(PROJECT_ROOT)),
}

{'project_root': '/Users/mac/Documents/Project/open_cancer',
 'train': 'data/raw/train.csv',
 'test': 'data/raw/test.csv',
 'fold': 'data/splits/stratified_5fold_seed42.csv',
 'features': 'data/processed/mutation_type_features'}

## 2. 원본 데이터 계약과 공용 fold 검증

In [6]:
data_summary = validate_competition_data(
    TRAIN_PATH,
    TEST_PATH,
    SAMPLE_SUBMISSION_PATH,
)

train_meta = pd.read_csv(TRAIN_PATH, usecols=["ID", "SUBCLASS"], dtype=str)
test_meta = pd.read_csv(TEST_PATH, usecols=["ID"], dtype=str)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH, dtype=str, keep_default_na=False)
folds = pd.read_csv(FOLD_PATH, dtype={"ID": str, "fold": int})

train = train_meta.merge(folds, on="ID", how="left", validate="one_to_one", sort=False)
if not train["ID"].equals(train_meta["ID"]):
    raise ValueError("fold 병합 과정에서 train 행 순서가 바뀌었습니다.")
if train["fold"].isna().any() or set(train["fold"]) != set(range(N_SPLITS)):
    raise ValueError("공용 fold가 모든 train ID에 0~4로 배정되지 않았습니다.")
if not sample_submission["ID"].equals(test_meta["ID"]):
    raise ValueError("sample_submission과 test ID 값 또는 순서가 다릅니다.")

display(pd.DataFrame({
    "항목": ["train rows", "test rows", "gene columns", "class count"],
    "값": [data_summary["train_rows"], data_summary["test_rows"], data_summary["gene_columns"], data_summary["class_count"]],
}))
train["fold"].value_counts().sort_index()

,항목,값
0,train rows,6201
1,test rows,2546
2,gene columns,4384
3,class count,26


fold
0    1241
1    1240
2    1240
3    1240
4    1240
Name: count, dtype: int64

## 3. 변이 유형 희소 피처 생성

피처 생성기는 train의 `SUBCLASS`를 계산에 사용하지 않고 label 파일로만 분리합니다. 같은 유전자 순서와 고정 규칙을 train/test에 적용합니다.

In [7]:
feature_report = build_mutation_features(TRAIN_PATH, TEST_PATH, FEATURE_DIR)

x_all = sparse.load_npz(FEATURE_DIR / "train_features.npz")
x_test_shared = sparse.load_npz(FEATURE_DIR / "test_features.npz")
feature_names = json.loads((FEATURE_DIR / "feature_names.json").read_text(encoding="utf-8"))
feature_train_ids = pd.read_csv(FEATURE_DIR / "train_ids.csv", dtype=str)
feature_test_ids = pd.read_csv(FEATURE_DIR / "test_ids.csv", dtype=str)
feature_labels = pd.read_csv(FEATURE_DIR / "train_labels.csv", dtype=str)

if not feature_train_ids["ID"].equals(train["ID"]):
    raise ValueError("피처 train ID 순서가 원본과 다릅니다.")
if not feature_test_ids["ID"].equals(test_meta["ID"]):
    raise ValueError("피처 test ID 순서가 원본과 다릅니다.")
if not feature_labels["SUBCLASS"].equals(train["SUBCLASS"]):
    raise ValueError("분리 저장된 label 순서가 원본과 다릅니다.")
if not np.isfinite(x_all.data).all() or not np.isfinite(x_test_shared.data).all():
    raise ValueError("피처 행렬에 NaN 또는 무한대가 있습니다.")

feature_report

{'inputs': {'train': {'path': '/Users/mac/Documents/Project/open_cancer/data/raw/train.csv',
   'sha256': '92418b8441d058cfc68e939dd88725610750be4bc8edc51253cffc72fc4fc0ab'},
  'test': {'path': '/Users/mac/Documents/Project/open_cancer/data/raw/test.csv',
   'sha256': 'e7e7f29a9b6251308e470ae3fb040a6da0cd8fcb0adb87e67f7761631c6a1ef0'}},
 'feature_contract': {'target_used_for_features': False,
  'gene_count': 4384,
  'gene_order_sha256': 'fa63b715c465a557b42670e8563ac4dee1bd6d8378cf8c500dfcbda72bc436ff',
  'mutation_types': ['missense',
   'synonymous',
   'nonsense',
   'frameshift',
   'complex'],
  'missing_policy': 'separate per-gene and per-sample missing indicators',
  'position_features': 'excluded because a reliable source transcript/protein length is unavailable'},
 'train': {'shape': [6201, 30697], 'nonzero': 481761},
 'test': {'shape': [2546, 30697], 'nonzero': 414470},
 'feature_count': 30697,
 'feature_names_sha256': 'a52489aebcc6f50b48d93067c84063c009ea4a370444d8cfa3e8a390

## 4. pandas로 전처리 결과 확인

전체 희소행렬을 dense DataFrame으로 바꾸지 않고, 9개 샘플 집계 피처만 변환합니다.

In [8]:
sample_indices = [
    index for index, name in enumerate(feature_names)
    if name.startswith("sample__")
]
sample_names = [feature_names[index] for index in sample_indices]

train_sample_features = pd.DataFrame(
    x_all[:, sample_indices].toarray(),
    columns=sample_names,
)
test_sample_features = pd.DataFrame(
    x_test_shared[:, sample_indices].toarray(),
    columns=sample_names,
)
train_overview = pd.concat([train[["ID", "SUBCLASS", "fold"]], train_sample_features], axis=1)

display(train_overview.head())
display(pd.DataFrame({
    "train_mean": train_sample_features.mean(),
    "test_mean": test_sample_features.mean(),
}))

,ID,SUBCLASS,fold,sample__mutated_gene_count,sample__total_variant_count,sample__multi_variant_gene_count,sample__missing_gene_count,sample__missense_count,sample__synonymous_count,sample__nonsense_count,sample__frameshift_count,sample__complex_count
0,TRAIN_0000,KIPAN,2,25.0,25.0,0.0,0.0,17.0,7.0,1.0,0.0,0.0
1,TRAIN_0001,SARC,4,17.0,17.0,0.0,0.0,9.0,5.0,0.0,3.0,0.0
2,TRAIN_0002,SKCM,1,119.0,124.0,5.0,0.0,79.0,39.0,6.0,0.0,0.0
3,TRAIN_0003,KIRC,0,7.0,7.0,0.0,0.0,3.0,3.0,1.0,0.0,0.0
4,TRAIN_0004,GBMLGG,0,34.0,34.0,0.0,0.0,23.0,11.0,0.0,0.0,0.0


,train_mean,test_mean
sample__mutated_gene_count,35.299629,78.134331
sample__total_variant_count,41.148846,132.565598
sample__multi_variant_gene_count,3.552008,24.814611
sample__missing_gene_count,0.000000,0.093087
sample__missense_count,26.566683,79.310684
sample__synonymous_count,10.785841,34.848389
sample__nonsense_count,2.143041,0.770228
sample__frameshift_count,1.598291,10.146111
sample__complex_count,0.054991,7.490180


In [9]:
def inspect_gene(gene: str, rows: int = 20) -> pd.DataFrame:
    indices = [
        index for index, name in enumerate(feature_names)
        if name.startswith(f"{gene}__")
    ]
    if not indices:
        raise KeyError(f"피처에 없는 유전자입니다: {gene}")
    gene_frame = pd.DataFrame(
        x_all[:, indices].toarray(),
        columns=[feature_names[index] for index in indices],
    )
    result = pd.concat([train[["ID", "SUBCLASS", "fold"]], gene_frame], axis=1)
    return result.loc[result[f"{gene}__mutated"].eq(1)].head(rows)


inspect_gene("TP53")

,ID,SUBCLASS,fold,TP53__mutated,TP53__missense,TP53__synonymous,TP53__nonsense,TP53__frameshift,TP53__complex,TP53__missing
1,TRAIN_0001,SARC,4,1.0,1.0,0.0,0.0,0.0,0.0,0.0
4,TRAIN_0004,GBMLGG,0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
13,TRAIN_0013,PAAD,4,1.0,0.0,0.0,0.0,1.0,0.0,0.0
14,TRAIN_0014,SKCM,3,1.0,0.0,0.0,1.0,0.0,0.0,0.0
21,TRAIN_0021,UCEC,3,1.0,1.0,0.0,0.0,0.0,0.0,0.0
31,TRAIN_0031,GBMLGG,3,1.0,1.0,0.0,0.0,0.0,0.0,0.0
32,TRAIN_0032,HNSC,1,1.0,0.0,0.0,1.0,0.0,0.0,0.0
36,TRAIN_0036,LGG,0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
37,TRAIN_0037,LUSC,2,1.0,0.0,0.0,1.0,0.0,0.0,0.0
38,TRAIN_0038,LUSC,1,1.0,1.0,0.0,0.0,0.0,0.0,0.0


## 5. 타깃 인코딩

In [10]:
label_encoder = LabelEncoder()
label_encoder.fit(list(CLASS_LABELS))
if list(label_encoder.classes_) != list(CLASS_LABELS):
    raise ValueError("LabelEncoder 클래스 순서가 프로젝트 고정 순서와 다릅니다.")

y = label_encoder.transform(train["SUBCLASS"]).astype(np.int32)
pd.DataFrame({
    "SUBCLASS": label_encoder.classes_,
    "encoded": np.arange(len(label_encoder.classes_)),
})

,SUBCLASS,encoded
0,ACC,0
1,BLCA,1
2,BRCA,2
3,CESC,3
4,COAD,4
5,DLBC,5
6,GBMLGG,6
7,HNSC,7
8,KIPAN,8
9,KIRC,9


## 6. XGBoost 모델과 공용 5-fold 학습

기존 XGBoost Notebook과 동일한 모델 기본값을 사용합니다. 이번 비교에서 달라지는 핵심은 전처리 피처입니다.

In [ ]:
XGB_PARAMS = {
    "objective": "multi:softprob",
    "num_class": len(CLASS_LABELS),
    "n_estimators": 500,
    "learning_rate": 0.05,
    "max_depth": 6,
    "min_child_weight": 1.0,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "eval_metric": "mlogloss",
    "early_stopping_rounds": 30,
    "tree_method": "hist",
    "device": "cpu",
    "n_jobs": N_JOBS,
    "verbosity": 0,
}
XGB_PARAMS

In [ ]:
fold_results: list[dict[str, int | float | None]] = []
trained_models: list[xgb.XGBClassifier] = []
oof_proba = np.full((len(train), len(CLASS_LABELS)), np.nan, dtype=np.float32)
test_proba = np.zeros((len(test_meta), len(CLASS_LABELS)), dtype=np.float32)

if RUN_TRAINING:
    artifact_slug = f"{EXPERIMENT_ID.lower().replace('-', '')}_xgb_mutation_features"
    model_dir = PROJECT_ROOT / "models" / artifact_slug
    model_dir.mkdir(parents=True, exist_ok=True)

    for fold in range(N_SPLITS):
        valid_mask = train["fold"].eq(fold).to_numpy()
        train_indices = np.flatnonzero(~valid_mask)
        valid_indices = np.flatnonzero(valid_mask)

        x_train_fold = x_all[train_indices]
        x_valid_fold = x_all[valid_indices]
        y_train_fold = y[train_indices]
        y_valid_fold = y[valid_indices]
        sample_weight = (
            compute_sample_weight(class_weight="balanced", y=y_train_fold)
            if USE_BALANCED_SAMPLE_WEIGHT
            else None
        )

        model = xgb.XGBClassifier(**XGB_PARAMS, random_state=SEED + fold)
        model.fit(
            x_train_fold,
            y_train_fold,
            sample_weight=sample_weight,
            eval_set=[(x_valid_fold, y_valid_fold)],
            verbose=False,
        )
        if not np.array_equal(model.classes_, np.arange(len(CLASS_LABELS))):
            raise ValueError(f"fold {fold} 모델의 확률 클래스 순서가 다릅니다.")

        valid_proba = model.predict_proba(x_valid_fold).astype(np.float32)
        oof_proba[valid_indices] = valid_proba
        test_proba += model.predict_proba(x_test_shared).astype(np.float32) / N_SPLITS

        fold_macro_f1 = f1_score(y_valid_fold, valid_proba.argmax(axis=1), average="macro")
        best_iteration = getattr(model, "best_iteration", None)
        fold_results.append({
            "fold": fold,
            "macro_f1": float(fold_macro_f1),
            "best_iteration": None if best_iteration is None else int(best_iteration),
            "train_rows": len(train_indices),
            "valid_rows": len(valid_indices),
        })
        model.save_model(model_dir / f"fold_{fold:02d}.json")
        trained_models.append(model)
        print(f"fold={fold} macro_f1={fold_macro_f1:.6f} best_iteration={best_iteration}")

    if np.isnan(oof_proba).any():
        raise ValueError("OOF 확률에 채워지지 않은 행이 있습니다.")
else:
    print("RUN_MODE='explore'이므로 전처리까지만 실행하고 모델 학습은 건너뜁니다.")

## 7. OOF Macro F1 평가

In [ ]:
if RUN_TRAINING:
    oof_predictions = oof_proba.argmax(axis=1)
    oof_macro_f1 = f1_score(y, oof_predictions, average="macro")
    class_report = classification_report(
        y,
        oof_predictions,
        labels=np.arange(len(CLASS_LABELS)),
        target_names=CLASS_LABELS,
        output_dict=True,
        zero_division=0,
    )
    fold_metrics = pd.DataFrame(fold_results)
    class_f1 = pd.DataFrame({
        "SUBCLASS": CLASS_LABELS,
        "f1": [class_report[label]["f1-score"] for label in CLASS_LABELS],
        "support": [int(class_report[label]["support"]) for label in CLASS_LABELS],
    })
    print(f"전체 OOF Macro F1: {oof_macro_f1:.6f}")
    display(fold_metrics)
    display(class_f1)
else:
    print("측정된 모델 점수는 없습니다.")

## 8. 테스트 추론과 제출 파일 생성

In [ ]:
if RUN_TRAINING:
    artifact_slug = f"{EXPERIMENT_ID.lower().replace('-', '')}_xgb_mutation_features"
    oof_dir = PROJECT_ROOT / "oof"
    preds_dir = PROJECT_ROOT / "preds"
    submissions_dir = PROJECT_ROOT / "submissions"
    report_dir = PROJECT_ROOT / "reports" / artifact_slug
    for directory in (oof_dir, preds_dir, submissions_dir, report_dir):
        directory.mkdir(parents=True, exist_ok=True)

    oof_predictions = oof_proba.argmax(axis=1)
    oof_frame = pd.DataFrame({
        "ID": train["ID"],
        "SUBCLASS_TRUE": train["SUBCLASS"],
        "SUBCLASS_PRED": label_encoder.inverse_transform(oof_predictions),
        "FOLD": train["fold"].astype(int),
    })
    oof_frame.loc[:, list(PROBABILITY_COLUMNS)] = oof_proba

    test_probability_frame = pd.DataFrame({"ID": test_meta["ID"]})
    test_probability_frame.loc[:, list(PROBABILITY_COLUMNS)] = test_proba
    submission = sample_submission.copy()
    submission["SUBCLASS"] = label_encoder.inverse_transform(test_proba.argmax(axis=1))

    oof_path = oof_dir / f"{artifact_slug}.csv"
    test_probability_path = preds_dir / f"{artifact_slug}_test_proba.csv"
    submission_path = submissions_dir / f"{artifact_slug}.csv"
    oof_frame.to_csv(oof_path, index=False, lineterminator="\n")
    test_probability_frame.to_csv(test_probability_path, index=False, lineterminator="\n")
    submission.to_csv(submission_path, index=False, lineterminator="\n")

    submission_validation = validate_submission(submission_path, TEST_PATH)
    notebook_run_summary = {
        "issue_number": ISSUE_NUMBER,
        "experiment_id": EXPERIMENT_ID,
        "branch": experiment_context.branch,
        "seed": SEED,
        "feature_contract": feature_report["feature_contract"],
        "feature_names_sha256": feature_report["feature_names_sha256"],
        "balanced_sample_weight": USE_BALANCED_SAMPLE_WEIGHT,
        "xgboost_version": xgb.__version__,
        "xgb_params": XGB_PARAMS,
        "fold_results": fold_results,
        "oof_macro_f1": float(oof_macro_f1),
        "data_files": data_summary["files"],
        "submission_validation": submission_validation,
    }
    summary_path = report_dir / "notebook_run.json"
    summary_path.write_text(json.dumps(notebook_run_summary, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"OOF: {oof_path.relative_to(PROJECT_ROOT)}")
    print(f"Test probabilities: {test_probability_path.relative_to(PROJECT_ROOT)}")
    print(f"Submission: {submission_path.relative_to(PROJECT_ROOT)}")
    print(f"Summary: {summary_path.relative_to(PROJECT_ROOT)}")
    display(submission.head())
    submission_validation
else:
    print("학습하지 않았으므로 OOF, checkpoint와 제출 파일을 만들지 않았습니다.")

## 9. 공식 실험으로 전환할 때

1. 현재 Issue가 `experiment` label을 가진 공식 Experiment Issue인지 확인합니다.
2. 아니라면 새 Experiment Issue와 해당 번호의 브랜치를 만듭니다.
3. Notebook에서 확인한 로직을 `configs/`와 `scripts/run_expNNN_<slug>.py`로 옮깁니다.
4. 기본값과 override가 합쳐진 `config.resolved.yaml`을 저장합니다.
5. 공용 split 전체 OOF와 Macro F1을 측정한 뒤 실제 결과만 History에 기록합니다.